# Semantic Similarity Backup Layer — Real CodeBERT Validation

Companion notebook to the structural (AST-only) competency extraction paper.

**Purpose:** validate the semantic-similarity backup signal against real Refactory
submissions, using real `microsoft/codebert-base` embeddings instead of the
bag-of-AST-tokens stub used in the sandboxed development environment (which was
shown to have no discriminative power).

**Non-negotiable design principle carried over from the structural layer:**
semantic similarity NEVER modifies `final_score`. It is a separate, labeled,
unverified signal for manual review only.

**What this notebook does, step by step:**
1. Install dependencies and confirm GPU/CPU availability
2. Upload your project files and Refactory data
3. Build real CodeBERT embeddings (replaces the `stub` backend)
4. Build a proper exemplar registry from real reference solutions
5. Build the actual test set that matters: submissions that are test-suite-correct
   but scored 0 (or near-0) structurally — the cases this layer exists to catch
6. Compute similarity, sweep thresholds, and report honestly — including if it
   *doesn't* beat the stub baseline


## 1. Setup

Uses the latest `transformers`/`torch`. (An earlier version of this notebook
pinned an older `transformers` to work around a config-loading bug in
`codet5p-110m-embedding`'s remote code -- that pin turned out to drag in an
old `tokenizers` release with no prebuilt wheel for Colab's current Python,
which fails to build from source without a Rust toolchain. Section 3 below
fixes the original bug a different way that doesn't need any pin.)


In [ ]:
!pip install -q transformers torch pandas scikit-learn matplotlib

import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", "cuda" if torch.cuda.is_available() else "cpu (fine for codet5p-110m-embedding on short snippets)")

import transformers
print("transformers version:", transformers.__version__)


## 2. Get your files into this session

Two options. **Drive (recommended)** — set up once, never re-upload again.
**Upload** — quick one-off, but you re-upload every new Colab session since
Colab's local disk doesn't persist between sessions.

### Option A — Google Drive (persists across sessions)

One-time setup, in Google Drive (not in Colab):

1. In My Drive, create a folder, e.g. `competency_extraction/`
2. Inside it, put:
   ```
   competency_extraction/
     competency.py
     semantic_backup.py
     topk_report.csv              (or whichever batch CSV)
     refactory/
       data/
         question_5/code/correct/*.py
         question_5/code/wrong/*.py
         ... (other questions as needed)
   ```
3. Run the mount cell below — first time only, it'll ask you to authorize
   access via a popup/link. After that, every new notebook session just needs
   the mount cell re-run (no re-authorization, no re-upload).

`PROJECT_DIR` below is used by every later cell in this notebook — change it
once here rather than editing paths throughout.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_DIR = '/content/drive/MyDrive/competency_extraction'  # <-- adjust to your actual folder name/path

import os
assert os.path.isdir(PROJECT_DIR), f"Not found: {PROJECT_DIR} -- check the folder name/path in Drive"
print("Contents:", os.listdir(PROJECT_DIR))


### Option B — one-off upload instead

Skip this if you used Option A above. Only needed if you don't want to set up
a Drive folder and are OK re-uploading each session.


In [ ]:
# Only run this if you're using Option B instead of Drive.
# from google.colab import files
# uploaded = files.upload()  # select competency.py, semantic_backup.py, topk_report.csv, etc.
# PROJECT_DIR = '.'  # uploaded files land in the current working directory
# print(list(uploaded.keys()))


## 3. Real embeddings -- and why raw CodeBERT alone isn't enough

**If you already ran an earlier version of this cell and got something like**
`related vs related: 0.96` **and** `related vs unrelated: 0.96` **(i.e. no
separation, or unrelated even scoring *higher*) -- that's not a bug in your
setup. It's a well-documented property called **anisotropy**: raw,
non-fine-tuned transformer embeddings (BERT, RoBERTa, and `codebert-base`
included) collapse into a narrow cone in embedding space when mean-pooled or
`[CLS]`-pooled, so nearly *everything* ends up with high cosine similarity to
everything else, regardless of actual content. This is documented in the
literature (Ethayarajh 2019; Li et al. 2020) and reproduced directly by other
users hitting this exact symptom with `codebert-base`
(see [microsoft/CodeBERT issue #13](https://github.com/microsoft/CodeBERT/issues/13)).
`codebert-base` was trained with masked-language-modeling and
replaced-token-detection objectives -- neither one makes pooled-embedding
cosine similarity meaningful on its own.

**The fix:** use a model actually trained with a *contrastive* objective for
code retrieval/similarity, the same way Sentence-BERT fixed this for regular
BERT. This notebook uses
[`Salesforce/codet5p-110m-embedding`](https://huggingface.co/Salesforce/codet5p-110m-embedding) --
CodeT5+'s dedicated embedding checkpoint (encoder + projection layer),
pretrained with contrastive learning and text-code matching specifically for
this purpose. It reports 74.23% average retrieval accuracy across 6 languages
(including Python) on CodeXGLUE code-search benchmarks -- a citable number for
the companion paper -- and is small (110M params).

Note on usage: this checkpoint is a fixed, dedicated embedding model with its
own `custom_code` wrapper -- it does **not** use CodeT5+'s general
`<encoder_only>`-style mode-prefix tokens (those apply to the general
`codet5p-220m`/`770m` checkpoints, not this one). Passing raw code straight
into the tokenizer, as below, is the officially documented usage.


In [ ]:
from transformers import AutoConfig, AutoModel, AutoTokenizer
import torch

_checkpoint = "Salesforce/codet5p-110m-embedding"
_device = "cuda" if torch.cuda.is_available() else "cpu"

_tokenizer = AutoTokenizer.from_pretrained(_checkpoint, trust_remote_code=True)

# codet5p-110m-embedding's remote config code (written in 2023) never calls
# super().__init__(), so several base PretrainedConfig attributes are never
# actually set on the instance. Older transformers tolerated this via
# class-level defaults; recent transformers enforces attributes strictly and
# raises AttributeError (e.g. "'CodeT5pEmbeddingConfig' object has no
# attribute 'is_decoder'"). Patching the missing attributes directly, right
# after loading the config, fixes this without pinning any library version.
_config = AutoConfig.from_pretrained(_checkpoint, trust_remote_code=True)
for _attr, _default in [("is_decoder", False), ("is_encoder_decoder", True),
                         ("pad_token_id", 0), ("eos_token_id", 2)]:
    if not hasattr(_config, _attr):
        setattr(_config, _attr, _default)

_model = AutoModel.from_pretrained(_checkpoint, config=_config, trust_remote_code=True).to(_device)
_model.eval()

@torch.no_grad()
def codebert_embed(code: str):
    # kept the same function name so the rest of the notebook doesn't change
    inputs = _tokenizer.encode(code, return_tensors="pt", truncation=True, max_length=512).to(_device)
    embedding = _model(inputs)[0]  # already L2-normalized: norm == 1.0
    return embedding.cpu().numpy()

def cosine_sim(a, b) -> float:
    import numpy as np
    # embeddings are already normalized, so dot product == cosine similarity
    return float(np.dot(a, b))

# Sanity check: this should show a real gap, not ~0.96 vs ~0.96
related_a = "def f(lst):\n    best = lst[0]\n    for x in lst:\n        if x > best:\n            best = x\n    return best"
related_b = "def f(lst):\n    return max(lst)"
unrelated  = "def greet(name):\n    return 'Hello, ' + name"

print("related vs related:", cosine_sim(codebert_embed(related_a), codebert_embed(related_b)))
print("related vs unrelated:", cosine_sim(codebert_embed(related_a), codebert_embed(unrelated)))


## 4. Real exemplar registry

Replace the 2–3 hand-written snippets from the sandboxed version with real
reference solutions pulled from your Refactory `correct/` folders — a handful
of *diverse* correct submissions per competency, not just one canonical style.

Paste in real file paths / snippets below. This is the single biggest lever on
result quality — a thin or homogeneous registry will under- or over-rate
everything compared against it.


In [ ]:
import glob

def load_exemplars(folder_glob: str, n: int = 6):
    """Grab n real correct submissions to use as exemplars for one competency."""
    paths = sorted(glob.glob(folder_glob))[:n]
    return [open(p, encoding="utf-8").read() for p in paths]

EXEMPLARS = {
    # Example — adjust paths to your actual uploaded/mounted data
    "FindExtremum": load_exemplars(f"{PROJECT_DIR}/refactory/data/question_5/code/correct/*.py", n=6),
    "Accumulation": load_exemplars(f"{PROJECT_DIR}/refactory/data/question_2/code/correct/*.py", n=6),
    # add one entry per competency you want semantic backup for
}

EXEMPLAR_EMBEDDINGS = {
    cid: [codebert_embed(snippet) for snippet in snippets]
    for cid, snippets in EXEMPLARS.items()
}
print({cid: len(v) for cid, v in EXEMPLAR_EMBEDDINGS.items()})


## 5. Build the test set that actually matters

Not "everything" — specifically: submissions that are **test-suite-correct**
but scored **0 (or near-0) on the relevant structural competency**. These are
the genuine "structurally novel but correct" cases (e.g. a `functools.reduce`
based FindExtremum with no `ast.If`, no loop, no `max()`/`min()` call) — the
paradigm case this layer exists to catch.

Load your batch CSV report (from `batch_report.py`) and filter for exactly
this group, plus a comparison group of genuinely wrong submissions that also
score 0 structurally — you want to see whether real similarity separates
"novel-correct" from "actually-wrong", not just "similar to exemplar" in
general.


In [ ]:
import pandas as pd

report = pd.read_csv(f"{PROJECT_DIR}/topk_report.csv")  # or whichever batch CSV you uploaded

COMPETENCY = "FindExtremum"  # adjust to the column you're validating

novel_correct = report[
    (report["label"] == "correct") &
    (report[f"{COMPETENCY}_final"] == 0)
]
genuinely_wrong = report[
    (report["label"] == "wrong") &
    (report[f"{COMPETENCY}_final"] == 0)
]

print(f"novel-correct candidates: {len(novel_correct)}")
print(f"genuinely-wrong comparison group: {len(genuinely_wrong)}")


## 6. Compute similarity and compare the two groups

If the semantic layer is doing real work, `novel_correct` similarity scores
should be *meaningfully* higher than `genuinely_wrong` scores against the same
exemplar set. If the two distributions overlap the way the stub's did
(within ~0.02 of each other), that's a real, honest negative result — report
it rather than picking a threshold that happens to look good on this sample.


In [ ]:
def max_similarity(code: str, competency_id: str) -> float:
    emb = codebert_embed(code)
    return max(cosine_sim(emb, ex) for ex in EXEMPLAR_EMBEDDINGS[competency_id])

def score_group(df, folder_prefix, competency_id):
    scores = []
    for fname in df["file"]:
        path = f"{folder_prefix}/{fname}"
        try:
            src = open(path, encoding="utf-8").read()
        except FileNotFoundError:
            continue
        scores.append(max_similarity(src, competency_id))
    return scores

# adjust folder_prefix to wherever the actual .py files live
novel_scores = score_group(novel_correct, f"{PROJECT_DIR}/refactory/data/question_5/code/correct", COMPETENCY)
wrong_scores  = score_group(genuinely_wrong, f"{PROJECT_DIR}/refactory/data/question_5/code/wrong", COMPETENCY)

import numpy as np
print(f"novel-correct: mean={np.mean(novel_scores):.3f} std={np.std(novel_scores):.3f} n={len(novel_scores)}")
print(f"genuinely-wrong: mean={np.mean(wrong_scores):.3f} std={np.std(wrong_scores):.3f} n={len(wrong_scores)}")


## 7. Threshold sweep

Pick a defensible cutoff from real data rather than assuming `0.5`. This uses
ROC/AUC framing: can `max_similarity` separate "novel-correct" (positive) from
"genuinely-wrong" (negative)?


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib.pyplot as plt

y_true = [1] * len(novel_scores) + [0] * len(wrong_scores)
y_score = novel_scores + wrong_scores

auc = roc_auc_score(y_true, y_score)
fpr, tpr, thresholds = roc_curve(y_true, y_score)

print(f"AUC: {auc:.3f}  (0.5 = no better than chance / the stub's failure mode)")

plt.figure(figsize=(5,5))
plt.plot(fpr, tpr, label=f"CodeBERT (AUC={auc:.2f})")
plt.plot([0,1],[0,1],'--', color='gray', label="chance")
plt.xlabel("False positive rate (genuinely-wrong flagged)")
plt.ylabel("True positive rate (novel-correct flagged)")
plt.legend()
plt.title(f"Semantic backup separability — {COMPETENCY}")
plt.show()

# Pick threshold nearest a target true-positive rate, e.g. 0.8
import numpy as np
target_tpr = 0.8
idx = np.argmin(np.abs(tpr - target_tpr))
print(f"Suggested threshold for ~{target_tpr:.0%} recall: {thresholds[idx]:.3f} "
      f"(fpr at this point: {fpr[idx]:.3f})")


## 8. Save results

Export the raw scores and the chosen threshold so they can go straight into
the companion paper's Results section, with the same honesty standard as the
structural paper's Table 4 — report the AUC and threshold you actually got,
not the one you hoped for.


In [ ]:
import pandas as pd

pd.DataFrame({
    "file": list(novel_correct["file"]) [:len(novel_scores)] + list(genuinely_wrong["file"])[:len(wrong_scores)],
    "group": ["novel_correct"] * len(novel_scores) + ["genuinely_wrong"] * len(wrong_scores),
    "similarity": novel_scores + wrong_scores,
}).to_csv(f"{PROJECT_DIR}/semantic_backup_validation_results.csv", index=False)  # saved back to Drive too

# Also grab a local copy in this session, if you want it immediately:
from google.colab import files
files.download(f"{PROJECT_DIR}/semantic_backup_validation_results.csv")
